# Data and Methodology

## Data

Data was collected across three countries: Australia (treatment), the United Kingdom (control), and New Zealand (control). Additional enrolment data was collected for Canada as a potential extra control, however it was ultimately excluded from the primary analysis as it introduced excess noise into the panel and was instead retained as an auxiliary robustness check. Panel data was utilised for all three countries.

Three categories of data were collected. First, university enrolments for all three countries spanning 2016–2024 across all disciplines and fields of study, which served as the centrepiece of the difference-in-differences analysis. Second, Australian job market shortage data across a range of industries, which was excluded from the primary econometric analysis and used instead to provide contextual support for the classification of disciplines as high-priority or low-priority under the JRGS. Third, tuition fees and commonwealth funding clusters for Australian tertiary disciplines across 2016–2024, which were matched to their respective disciplines to quantify how student contributions and government co-payments changed as a result of the reform.

## Data Cleaning and Sample Restrictions

Disciplinary enrolments across the three countries were assigned common category keys to ensure each Australian treatment discipline had a directly comparable control counterpart in both the UK and New Zealand. The UK HESA data reported enrolments by academic year (e.g. 2016/2017), creating a six-month misalignment with the Australian and New Zealand calendar-year data; this was resolved by mapping each academic year to its start year (e.g. 2016/2017 became 2016), producing a consistent annual time index across all three countries. Australian funding clusters were also standardised across all time periods, as different versions of the raw data classified certain disciplines across different clusters; these were corrected to ensure that funding figures accurately reflected each discipline's actual allocation throughout the study window. The same category key system was then used to link funding clusters to their corresponding disciplines.

Certain disciplines were not consistently categorised in the raw data for the years 2016–2018, particularly for UK subject classifications which changed in 2019/2020. Observations for these disciplines prior to 2019 were dropped to preserve comparability, limiting five disciplines — natural and physical sciences, environment and related, society and culture, others, and architecture and building — to a 2019–2024 panel. The final observation count per discipline is N=27 (3 countries × 9 years) for disciplines with the full 2016–2024 panel, and N=18 (3 countries × 6 years) for disciplines restricted to 2019–2024. A log transformation was applied to the enrolment outcome variable to express results as percentage changes and to accommodate the substantial differences in nominal enrolment scale across the three countries. Together, these adjustments produce a balanced and comparable panel in which the primary source of structural variation is the JRGS shock rather than differences in national counting conventions.

### Treatment Intensity by Discipline

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

FUND_PATH = ROOT / 'data' / 'clean' / 'AnnualFundingAUS2019-2026_with_category_key.csv'
fund = pd.read_csv(FUND_PATH)
fund['MaximumStudentContribution'] = pd.to_numeric(fund['MaximumStudentContribution'], errors='coerce')
fund['CommonwealthContribution']   = pd.to_numeric(fund['CommonwealthContribution'],   errors='coerce')

DISCIPLINES = {
    1: 'Natural & Physical Science', 2: 'Information Technology',
    3: 'Engineering & Related Tech',  4: 'Architecture & Building',
    5: 'Environment & Related',       6: 'Health',
    7: 'Education',                   8: 'Management & Commerce',
    9: 'Society & Culture',          10: 'Creative Arts',
}
JRGS = {
    1: 'Priority', 2: 'Priority', 3: 'Priority', 4: 'Priority',
    5: 'Priority', 6: 'Priority', 7: 'Priority',
    8: 'Non-priority', 9: 'Non-priority', 10: 'Non-priority',
}

pre    = fund[fund['Year'].isin([2019, 2020])].groupby('CategoryKey')['MaximumStudentContribution'].mean()
post21 = fund[fund['Year'] == 2021].groupby('CategoryKey')['MaximumStudentContribution'].mean()
post24 = fund[fund['Year'] == 2024].groupby('CategoryKey')['MaximumStudentContribution'].mean()

rows = []
for key, name in DISCIPLINES.items():
    pre_v  = pre.get(key, np.nan)
    p21_v  = post21.get(key, np.nan)
    p24_v  = post24.get(key, np.nan)
    chg_abs = p21_v - pre_v
    chg_pct = (p21_v / pre_v - 1) * 100 if pre_v > 0 else np.nan
    rows.append({
        'Discipline':        name,
        'JRGS':              JRGS[key],
        'Pre-JRG (avg 2019–20)': f'${pre_v:,.0f}' if not np.isnan(pre_v) else '—',
        '2021':              f'${p21_v:,.0f}' if not np.isnan(p21_v) else '—',
        '2024':              f'${p24_v:,.0f}' if not np.isnan(p24_v) else '—',
        'Change ($)':        f'{chg_abs:+,.0f}' if not np.isnan(chg_abs) else '—',
        'Change (%)':        f'{chg_pct:+.1f}%' if not np.isnan(chg_pct) else '—',
    })

tbl = (pd.DataFrame(rows)
         .set_index('Discipline')
         .sort_values(['JRGS', 'Change (%)'], ascending=[True, True]))

print('Table. Maximum Student Contribution by Discipline — Pre- and Post-JRGS (AUD per EFTSL)')
print('Source: Australian Government Department of Education, Annual Funding 2019–2026')
print()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(tbl.to_string())

The JRGS produced a wide dispersion in fee changes across disciplines that had previously been relatively compressed. Before the reform, maximum student contributions ranged from approximately $6,500 to $11,000 per equivalent full-time student load. By 2021, that range had widened to roughly $4,000–$15,000, with priority disciplines receiving cuts of between 20% and 46% and non-priority disciplines absorbing increases of 28–36%. Education received the largest proportional cut among priority fields, while management and commerce faced the steepest increase among non-priority fields — the same two disciplines that produce the only statistically significant results in the analysis. The variation in treatment intensity across disciplines provides a natural dose-response test: if fee signals drive enrolment decisions, the largest effects should be concentrated in the disciplines with the largest fee changes, which is broadly consistent with the findings.

### Summary Statistics

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

AUS_PATH  = ROOT / 'data' / 'clean' / 'EnrollmentsAUS_category_with_numeric_key.csv'
UK_PATH   = ROOT / 'data' / 'clean' / 'uk_grouped' / 'with_categorykey' / 'UK_enrollments_grouped_comparison_all_years_with_categorykey.csv'
NZ_PATH   = ROOT / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv'

DISCIPLINES = {
    1:  'Natural & Physical Science', 2:  'Information Technology',
    3:  'Engineering & Related Tech',  4:  'Architecture & Building',
    5:  'Environment & Related',       6:  'Health',
    7:  'Education',                   8:  'Management & Commerce',
    9:  'Society & Culture',          10:  'Creative Arts',
    11: 'Others',
}
FULL_PANEL  = {3, 4, 6, 7, 8, 10}
SHORT_PANEL = {1, 2, 5, 9, 11}

aus_raw = pd.read_csv(AUS_PATH)
year_cols = [c for c in aus_raw.columns if str(c).isdigit()]
aus_long = aus_raw.melt(id_vars=['Category','CategoryKey'], value_vars=year_cols,
                         var_name='year', value_name='enrollments')
aus_long['year'] = aus_long['year'].astype(int)
aus_long['enrollments'] = pd.to_numeric(aus_long['enrollments'], errors='coerce')
aus_long['country'] = 'AUS'
aus_long = aus_long.rename(columns={'CategoryKey': 'category_key'})

uk_raw = pd.read_csv(UK_PATH)
uk_raw['year'] = uk_raw['AcademicYear'].str[:4].astype(int)
uk_raw['enrollments'] = pd.to_numeric(uk_raw['Total UK'], errors='coerce')
uk_raw['category_key'] = uk_raw['categorykey']
uk_raw['country'] = 'UK'

nz_raw = pd.read_csv(NZ_PATH)
nz_raw['enrollments'] = pd.to_numeric(nz_raw['total_bachelors'], errors='coerce')
nz_raw['country'] = 'NZ'

rows = []
for key, name in DISCIPLINES.items():
    yr_min = 2019 if key in SHORT_PANEL else 2016
    for country, df, kcol in [
        ('AUS', aus_long, 'category_key'),
        ('UK',  uk_raw,   'category_key'),
        ('NZ',  nz_raw,   'category_key'),
    ]:
        sub = df[(df[kcol] == key) & (df['year'].between(yr_min, 2024))][['year','enrollments','country']].copy()
        sub['discipline'] = name
        sub['category_key'] = key
        rows.append(sub)

panel = pd.concat(rows, ignore_index=True).dropna(subset=['enrollments'])
panel['log_enrollments'] = np.log(panel['enrollments'])
panel['post']    = (panel['year'] >= 2021).astype(int)
panel['treated'] = (panel['country'] == 'AUS').astype(int)

rows_out = []
for key in sorted(DISCIPLINES.keys()):
    name = DISCIPLINES[key]
    sub  = panel[panel['category_key'] == key]
    enr  = sub['enrollments']
    log_ = sub['log_enrollments']
    rows_out.append({
        'Discipline':         name,
        'Panel':              'Full (2016–2024)' if key not in SHORT_PANEL else 'Short (2019–2024)',
        'N':                  len(sub),
        'Mean enrolments':    f"{enr.mean():,.0f}",
        'SD enrolments':      f"{enr.std():,.0f}",
        'Mean log(enrol)':    f"{log_.mean():.3f}",
        'SD log(enrol)':      f"{log_.std():.3f}",
        'Pre-2021 mean log':  f"{sub[sub['post']==0]['log_enrollments'].mean():.3f}",
        'Post-2021 mean log': f"{sub[sub['post']==1]['log_enrollments'].mean():.3f}",
    })

sumstats = pd.DataFrame(rows_out).set_index('Discipline')
print('Table 0. Summary Statistics — Enrolment Panel (AUS, UK, NZ; all disciplines)')
print('Units: enrolments = headcount; log(enrolments) = natural log of annual headcount')
print()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(sumstats.to_string())
print()
print(f'Total observations: {len(panel):,}  |  Countries: AUS (treated), UK (control), NZ (control)')
print(f'Full panel (N=27 each): {", ".join(DISCIPLINES[k] for k in sorted(FULL_PANEL))}')
print(f'Short panel (N=18 each): {", ".join(DISCIPLINES[k] for k in sorted(SHORT_PANEL))}')

### Sample Construction and Exclusions

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

AUS_PATH = ROOT / 'data' / 'clean' / 'EnrollmentsAUS_category_with_numeric_key.csv'
UK_PATH  = ROOT / 'data' / 'clean' / 'uk_grouped' / 'with_categorykey' / 'UK_enrollments_grouped_comparison_all_years_with_categorykey.csv'
NZ_PATH  = ROOT / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv'

aus_raw = pd.read_csv(AUS_PATH)
uk_raw  = pd.read_csv(UK_PATH)
nz_raw  = pd.read_csv(NZ_PATH)

year_cols = [c for c in aus_raw.columns if str(c).isdigit()]
aus_long = aus_raw.melt(id_vars=['Category','CategoryKey'], value_vars=year_cols,
                         var_name='year', value_name='enrollments')
aus_long['year'] = aus_long['year'].astype(int)
aus_long['enrollments'] = pd.to_numeric(aus_long['enrollments'], errors='coerce')
uk_raw['year'] = uk_raw['AcademicYear'].str[:4].astype(int)
uk_raw['enrollments'] = pd.to_numeric(uk_raw['Total UK'], errors='coerce')
nz_raw['enrollments'] = pd.to_numeric(nz_raw['total_bachelors'], errors='coerce')

n_raw = len(aus_long) + len(uk_raw) + len(nz_raw)

aus_s1 = aus_long[aus_long['year'].between(2016, 2024)]
uk_s1  = uk_raw[uk_raw['year'].between(2016, 2024)]
nz_s1  = nz_raw[nz_raw['year'].between(2016, 2024)]
n_s1   = len(aus_s1) + len(uk_s1) + len(nz_s1)

KEYS   = list(range(1, 12))
aus_s2 = aus_s1[aus_s1['CategoryKey'].isin(KEYS)]
uk_s2  = uk_s1[uk_s1['categorykey'].isin(KEYS)]
nz_s2  = nz_s1[nz_s1['category_key'].isin(KEYS)]
n_s2   = len(aus_s2) + len(uk_s2) + len(nz_s2)

SHORT  = {1, 2, 5, 9, 11}
aus_s3 = aus_s2[~((aus_s2['CategoryKey'].isin(SHORT)) & (aus_s2['year'] < 2019))]
uk_s3  = uk_s2[~((uk_s2['categorykey'].isin(SHORT))   & (uk_s2['year'] < 2019))]
nz_s3  = nz_s2[~((nz_s2['category_key'].isin(SHORT))  & (nz_s2['year'] < 2019))]
n_s3   = len(aus_s3) + len(uk_s3) + len(nz_s3)

n_s4 = len(aus_s3.dropna(subset=['enrollments'])) + \
       len(uk_s3.dropna(subset=['enrollments']))  + \
       len(nz_s3.dropna(subset=['enrollments']))

lines = [
    f"Raw data collected: {n_raw:,} rows (AUS + UK + NZ)",
    "",
    f"  [−{n_raw - n_s1:>4}]  Restricted to 2016–2024 study window",
    f"  ────────────────────────────────────────────",
    f"            {n_s1:,}",
    "",
    f"  [−{n_s1 - n_s2:>4}]  Restricted to 11 matched category keys",
    f"  ────────────────────────────────────────────",
    f"            {n_s2:,}",
    "",
    f"  [−{n_s2 - n_s3:>4}]  Dropped pre-2019 for 5 UK-reclassified disciplines",
    f"            (Nat. Sci., IT, Env., Soc. & Culture, Others)",
    f"  ────────────────────────────────────────────",
    f"            {n_s3:,}",
    "",
    f"  [−{n_s3 - n_s4:>4}]  Dropped rows with missing enrolment values",
    f"  ────────────────────────────────────────────",
    f"  Final analytic sample: {n_s4:,} observations",
    "",
    f"    6 disciplines × 3 countries × 9 years = {6*3*9}  (full panel, N=27 each)",
    f"    5 disciplines × 3 countries × 6 years = {5*3*6}  (short panel, N=18 each)",
]
print('\n'.join(lines))

## Data Relation to Hypothesis

The hypothesis is that the JRGS fee changes shifted enrolments toward high-priority disciplines and away from those facing fee increases. The data structure is designed to isolate this effect: Australia is the only country in the panel to have altered its higher-education fee structure, while the UK and NZ held their funding arrangements effectively constant over the 2021–2024 period. The panel structure ensures that within-country, within-discipline variation over time drives the estimation rather than permanent scale differences between Australia and the control countries. The JRGS treatment also varies in intensity across disciplines — disciplines that experienced larger fee movements are expected to exhibit larger enrolment responses if a treatment effect exists. Year fixed effects absorb any global shocks common to all three countries in a given year, most notably the disruptions associated with COVID-19, leaving only the Australia-specific post-2021 divergence in enrolments to be attributed to the JRGS.

## Consistency of JRGS Fee Changes with Australian Job Shortages

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent
OSL_DIR = ROOT / 'EmploymentShortages'

GROUP23    = {'231': 4, '232': 3, '233': 3, '234': 1, '235': 3}
ANZSCO_MAP = {'11': 8, '12': 5, '13': 8, '14': 11, '21': 10, '22': 8, '24': 7, '25': 6, '26': 2, '27': 9}
CATEGORY_NAMES = {
    1: 'Natural & Physical Science', 2: 'Information Technology',
    3: 'Engineering & Related Tech',  4: 'Architecture & Building',
    5: 'Environment & Related',       6: 'Health',
    7: 'Education',                   8: 'Management & Commerce',
    9: 'Society & Culture',          10: 'Creative Arts',
}
JRGS_CLASS = {
    1: ('Priority','Decrease'), 2: ('Priority','Decrease'), 3: ('Priority','Decrease'),
    4: ('Priority','Decrease'), 5: ('Priority','Decrease'), 6: ('Priority','Decrease'),
    7: ('Priority','Decrease'), 8: ('Non-priority','Increase'),
    9: ('Non-priority','Increase'), 10: ('Non-priority','Increase'),
}

def code_to_key(code):
    s = str(int(code)).zfill(6)
    if s[:2] == '23': return GROUP23.get(s[:3])
    return ANZSCO_MAP.get(s[:2])

rows = []
for yr in [2021, 2022, 2023, 2024, 2025]:
    df  = pd.read_csv(OSL_DIR / f'OSL {yr} (ANZSCO 6).csv')
    sl1 = df[df['Skill Level'] == 1].copy()
    sl1['CategoryKey'] = sl1['Code'].apply(code_to_key)
    sl1 = sl1.dropna(subset=['CategoryKey'])
    sl1['CategoryKey'] = sl1['CategoryKey'].astype(int)
    for key, name in CATEGORY_NAMES.items():
        sub      = sl1[sl1['CategoryKey'] == key]
        n        = len(sub)
        shortage = (sub['National Shortage Rating'] == 'Shortage').sum()
        rows.append({'Year': yr, 'CategoryKey': key, 'Category': name,
                     'Shortage%': round(shortage / n * 100, 1) if n > 0 else 0.0})

data = pd.DataFrame(rows)
avg  = data.groupby(['CategoryKey','Category'])['Shortage%'].mean().round(1).reset_index()

table_rows = []
for key, name in CATEGORY_NAMES.items():
    priority, fee_dir = JRGS_CLASS[key]
    avg_val = avg[avg['CategoryKey'] == key]['Shortage%'].values
    avg_val = avg_val[0] if len(avg_val) else np.nan
    def yr_val(yr):
        v = data[(data['CategoryKey']==key) & (data['Year']==yr)]['Shortage%'].values
        return v[0] if len(v) else np.nan
    table_rows.append({
        'Discipline': name, 'JRGS': priority, 'Fee': fee_dir,
        '2021 (%)': yr_val(2021), '2023 (%)': yr_val(2023), '2025 (%)': yr_val(2025),
        'Avg 2021–25 (%)': avg_val,
        'Consistent?': 'Yes' if (priority=='Priority' and avg_val >= 20)
                             or (priority=='Non-priority' and avg_val < 20) else 'Partial / No',
    })

tbl = (pd.DataFrame(table_rows).set_index('Discipline')
         .sort_values(['JRGS', 'Avg 2021–25 (%)'], ascending=[True, False]))
print('Table. JRGS Fee Direction vs Australian Job Shortage Rates (Skill Level 1 Occupations)')
print('Source: Department of Employment and Workplace Relations, Occupation Shortage List 2021–2025')
print()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(tbl.to_string())

The JRGS fee structure is broadly consistent with the pattern of labour market shortages across the 2021–2025 period. Among the seven priority disciplines, health, engineering, education, and information technology all recorded persistently high shortage rates, peaking between 47% and 80% of Skill Level 1 occupations in national shortage at various points across the window. Among non-priority disciplines, management and commerce and creative arts show consistently low shortage rates (averaging under 15% and 1% respectively), supporting the rationale for the fee increases applied to those fields.

The alignment is not universal. Architecture and building, classified as a priority discipline, showed near-zero shortage rates until 2023 before rising sharply to 100% by 2025 — more consistent with a lagged cyclical demand shock than a structural supply shortage the policy was calibrated against. Society and culture, a non-priority field subject to fee increases, has seen a gradual upward trend driven primarily by legal and social work occupations. Nevertheless, for the disciplines central to this paper's findings — education and health on the priority side, management and commerce on the non-priority side — the fee changes are consistent with the labour market evidence, lending contextual support to the policy's stated objectives.

## Methodology

The estimating equation is a two-way fixed effects difference-in-differences model, estimated separately for each discipline:

$$\log E_{ct} = \alpha + \beta_1 \, \text{treated}_c + \beta_2 \, \text{NZ}_c + \beta_3 (\text{treated}_c \times \text{post}_t) + \sum_{t=2017}^{2024} \gamma_t \, \mathbf{1}[t] + \varepsilon_{ct}$$

where $c \in \{\text{AUS, UK, NZ}\}$ and $t$ indexes calendar years. $\text{treated}_c$ and $\text{NZ}_c$ are country indicators that absorb permanent level differences across the three countries, with the UK as the reference unit. $\text{post}_t = \mathbf{1}[t \geq 2021]$ marks the JRGS implementation period. Year fixed effects $\gamma_t$ absorb shocks common to all three countries in a given year, including the global disruption from COVID-19. The coefficient of interest is $\hat{\beta}_3$, which identifies the post-2021 deviation of Australian log-enrolments from the counterfactual trajectory implied by the pooled UK–NZ trend. The approximate percentage effect is $(e^{\hat{\beta}_3} - 1) \times 100$.

Standard errors are HC3 throughout. With G=3 country clusters, cluster-robust inference is not feasible — the Moulton factor for this panel implies non-trivial within-cluster error correlation, but the minimum cluster count for asymptotically reliable cluster-robust SEs is approximately 42. HC3 provides a conservative alternative. The primary specification leaves 15 residual degrees of freedom for full-panel disciplines (N=27) and 9 for short-panel disciplines (N=18).

The identifying assumption is that, absent the JRGS, Australian enrolments in each discipline would have followed a trend parallel to the pooled UK and NZ counterfactual from 2021 onward. This is evaluated below using pre-treatment trends for the six full-panel disciplines, where five years of pre-treatment data are available.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

AUS_PATH = ROOT / 'data' / 'clean' / 'EnrollmentsAUS_category_with_numeric_key.csv'
UK_PATH  = ROOT / 'data' / 'clean' / 'uk_grouped' / 'with_categorykey' / 'UK_enrollments_grouped_comparison_all_years_with_categorykey.csv'
NZ_PATH  = ROOT / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv'

FULL_PANEL = {
    3: 'Engineering & Related Tech',
    4: 'Architecture & Building',
    6: 'Health',
    7: 'Education',
    8: 'Management & Commerce',
   10: 'Creative Arts',
}
COLOURS = {'AUS': '#2166ac', 'UK': '#d6604d', 'NZ': '#4dac26'}

aus_raw = pd.read_csv(AUS_PATH)
year_cols = [c for c in aus_raw.columns if str(c).isdigit()]
aus_long = aus_raw.melt(id_vars=['Category','CategoryKey'], value_vars=year_cols,
                         var_name='year', value_name='enrollments')
aus_long['year'] = aus_long['year'].astype(int)
aus_long['enrollments'] = pd.to_numeric(aus_long['enrollments'], errors='coerce')
aus_long = aus_long.rename(columns={'CategoryKey': 'category_key'})
aus_long['country'] = 'AUS'

uk_raw = pd.read_csv(UK_PATH)
uk_raw['year'] = uk_raw['AcademicYear'].str[:4].astype(int)
uk_raw['enrollments'] = pd.to_numeric(uk_raw['Total UK'], errors='coerce')
uk_raw = uk_raw.rename(columns={'categorykey': 'category_key'})
uk_raw['country'] = 'UK'

nz_raw = pd.read_csv(NZ_PATH)
nz_raw['enrollments'] = pd.to_numeric(nz_raw['total_bachelors'], errors='coerce')
nz_raw['country'] = 'NZ'

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for idx, (key, name) in enumerate(FULL_PANEL.items()):
    ax = axes[idx]
    for country, df, kcol in [
        ('AUS', aus_long, 'category_key'),
        ('UK',  uk_raw,   'category_key'),
        ('NZ',  nz_raw,   'category_key'),
    ]:
        sub = (df[(df[kcol] == key) & (df['year'].between(2016, 2020))]
               [['year','enrollments']].dropna().sort_values('year').copy())
        sub['log_e'] = np.log(sub['enrollments'])
        base = sub[sub['year'] == 2016]['log_e'].values
        sub['delta'] = sub['log_e'] - (base[0] if len(base) else sub['log_e'].iloc[0])
        ax.plot(sub['year'], sub['delta'], 'o-', color=COLOURS[country],
                linewidth=1.8, markersize=5, label=country)

    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(name, fontsize=10)
    ax.set_xticks(range(2016, 2021))
    ax.set_ylabel('Cumulative log-change from 2016' if idx % 3 == 0 else '', fontsize=8)
    ax.tick_params(labelsize=8)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle('Pre-treatment log-enrolment trends — full-panel disciplines, indexed to 2016 (AUS, UK, NZ)',
             fontsize=11)
plt.tight_layout()
plt.show()

Pre-treatment trends are broadly parallel for most full-panel disciplines, though with meaningful variation. Management and commerce and health track closely across all three countries through 2016–2020, with deviations from a common slope remaining small throughout the pre-treatment window — these two disciplines provide the strongest support for the parallel trends assumption. Creative arts also shows close co-movement, though the small absolute enrolment numbers in NZ produce more volatility in the indexed series.

Education shows a mild but consistent upward divergence of AUS from the controls from 2017 onward, with Australian enrolments growing at a faster rate than both the UK and NZ in the pre-treatment period. While not large in magnitude, this pre-existing divergence is in the same direction as the positive post-treatment DiD estimate, which is consistent with the sign reversal observed when country-specific linear trends are added to the main specification. It suggests the education result may partly reflect a continuation of a pre-existing Australian trend rather than a discrete post-2021 response to the fee change. Engineering shows a similar pattern of AUS outpacing the controls from 2018, introducing analogous concerns for that discipline's estimate. Architecture is the most clearly parallel through 2019 but shows AUS enrolments plateauing while UK and NZ continue growing into 2020, which complicates the post-treatment counterfactual. The pre-treatment window is limited to 2019–2020 for the five short-panel disciplines, which is insufficient to meaningfully assess the parallel trends assumption for those fields.

## Data Limitations

### Raw Data

UK subject classifications changed in 2019/2020, rendering pre-2019 UK data irreconcilable with later figures for five disciplines. As a consequence, these disciplines enter the panel only from 2019, leaving just one pre-policy observation prior to the 2021 JRGS implementation. This makes it difficult to verify the parallel trends assumption for those disciplines and results in materially higher uncertainty in their DiD estimates relative to the six disciplines with the full 2016–2024 panel. Additionally, more specific sub-disciplines were aggregated into broader category groupings to harmonise data across countries — for example, management and commerce encompasses accounting, marketing, and finance, all of which were subject to the same funding changes. While this does not introduce inconsistencies, it reduces the specificity of the findings. The analysis is also constrained by a limited number of pre- and post-treatment observations, as enrolment data prior to 2016 was unavailable and the post-2021 window is inherently bounded by the current date. Finally, the JRGS applied only to domestic students, yet it was not possible to disaggregate domestic from international enrolments in the Australian source data, meaning the observed totals capture both groups. Since international students were unaffected by the commonwealth funding changes, their inclusion may attenuate the estimated treatment effects.

### Identification

The small number of control countries limits the analysis's resilience to country-specific shocks. With only the UK and NZ as comparators, any idiosyncratic shock to either control country in the post-treatment period could bias the DiD estimate. The same concern applies to the treatment side: with Australia as the sole treated unit, any Australia-specific time shock unrelated to the JRGS cannot be separated from the treatment effect. COVID-19 also had an asymmetric effect across countries; Australia's international border closures ran from 2020 to 2022, a longer period than either NZ or the UK, which may have differentially affected Australian enrolment numbers through reduced international student intake during the transition into the post-treatment window. Finally, while NZ provides a second control country and strengthens identification, its enrolment levels are substantially smaller than both Australia and the UK, which may give it a disproportionate influence on the pooled counterfactual despite its relatively small scale.

### Robustness and Inference

With only two primary control country clusters (G=2, or G=3 if Canada is included), cluster-robust standard errors are not feasible — the minimum number of clusters required for reliable cluster inference is approximately 42. HC3 heteroscedasticity-robust standard errors are used instead, justified by the Moulton factor. Permutation inference similarly cannot achieve conventional significance thresholds at this cluster size; with G=3 assignments, the minimum achievable permutation p-value is 1/3 ≈ 0.33. The permutation results serve as a model-free diagnostic rather than a formal significance test: for management and commerce, both placebo country assignments produce larger absolute DiD coefficients than the true Australian assignment (permutation p=1.00), while for education the true assignment ranks second (permutation p=0.67). The most substantive identification concern is that the education parallel trends assumption is not robust to country-specific linear trends — when country-specific trend terms are added to the main specification, the education DiD coefficient reverses sign from +0.171 to −0.043, suggesting that Australian education enrolments were on a diverging trajectory relative to the control countries prior to 2021, which undermines the reliability of that estimate.